In [1]:
import sys
print(sys.executable)  # shows which Python binary Jupyter is using
print(sys.version)

C:\Users\mphop\anaconda3\envs\allrank-env\python.exe
3.9.23 (main, Jun  5 2025, 13:25:08) [MSC v.1929 64 bit (AMD64)]


In [2]:
!dir 

 Volume in drive C has no label.
 Volume Serial Number is 52EA-68E8

 Directory of C:\Users\mphop\Documents\Jupyter Notebooks\Python exercises\main lightgbm model scripts\final code\final versions\allRank

08/13/2026  03:56 AM    <DIR>          .
08/12/2026  06:55 AM    <DIR>          ..
10/08/2025  08:25 PM               979 .gitignore
08/13/2026  03:52 AM    <DIR>          .ipynb_checkpoints
12/28/2025  12:56 PM    <DIR>          allrank
10/08/2025  08:47 PM    <DIR>          allRank.egg-info
08/13/2026  03:55 AM         1,364,395 allrank_listmle_male_combined_tensor_output.txt
08/12/2026  06:07 AM    <DIR>          athletes_men_data_combined
08/12/2026  06:07 AM    <DIR>          athletes_women_data_combined
10/08/2025  08:25 PM               482 Dockerfile
10/08/2025  08:56 PM    <DIR>          dummy_data
08/13/2026  03:56 AM            23,710 extract allRank from listmle male combined.ipynb
08/12/2026  06:25 AM            27,940 extract allRank from listmle women combined.ipynb
08

In [3]:
!dir dummy_data

 Volume in drive C has no label.
 Volume Serial Number is 52EA-68E8

 Directory of C:\Users\mphop\Documents\Jupyter Notebooks\Python exercises\main lightgbm model scripts\final code\final versions\allRank\dummy_data

10/08/2025  08:56 PM    <DIR>          .
08/13/2026  03:56 AM    <DIR>          ..
04/28/2025  08:00 PM           884,308 train.txt
04/28/2025  08:00 PM           884,016 vali.txt
               2 File(s)      1,768,324 bytes
               2 Dir(s)  198,292,209,664 bytes free


In [4]:
!dir 

 Volume in drive C has no label.
 Volume Serial Number is 52EA-68E8

 Directory of C:\Users\mphop\Documents\Jupyter Notebooks\Python exercises\main lightgbm model scripts\final code\final versions\allRank

08/13/2026  03:56 AM    <DIR>          .
08/12/2026  06:55 AM    <DIR>          ..
10/08/2025  08:25 PM               979 .gitignore
08/13/2026  03:52 AM    <DIR>          .ipynb_checkpoints
12/28/2025  12:56 PM    <DIR>          allrank
10/08/2025  08:47 PM    <DIR>          allRank.egg-info
08/13/2026  03:55 AM         1,364,395 allrank_listmle_male_combined_tensor_output.txt
08/12/2026  06:07 AM    <DIR>          athletes_men_data_combined
08/12/2026  06:07 AM    <DIR>          athletes_women_data_combined
10/08/2025  08:25 PM               482 Dockerfile
10/08/2025  08:56 PM    <DIR>          dummy_data
08/13/2026  03:56 AM            23,710 extract allRank from listmle male combined.ipynb
08/12/2026  06:25 AM            27,940 extract allRank from listmle women combined.ipynb
08

In [5]:
import json
config = json.load(open("scripts/athletes_listmle_config_listmle_women_combined.json"))
config["data"]["path"] = "athletes_women_data_combined"
json.dump(config, open("scripts/aathletes_listmle_config_listmle_women_combined_colab_config.json", "w"))

In [6]:
# train the model separately using docker and then copy the necessary files and use code below to get predictions

In [7]:
import os
from argparse import ArgumentParser, Namespace
from pprint import pformat
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import torch
from attr import asdict

from allrank.click_models.click_utils import click_on_slates
from allrank.config import Config
from allrank.data.dataset_loading import load_libsvm_dataset_role
from allrank.data.dataset_saving import write_to_libsvm_without_masked
from allrank.inference.inference_utils import rank_slates, metrics_on_clicked_slates
from allrank.models.model import make_model
from allrank.models.model_utils import get_torch_device, CustomDataParallel, load_state_dict_from_file
from allrank.utils.args_utils import split_as_strings
from allrank.utils.command_executor import execute_command
from allrank.utils.config_utils import instantiate_from_recursive_name_args
from allrank.utils.file_utils import create_output_dirs, PathsContainer, copy_local_to_gs
from allrank.utils.ltr_logging import init_logger
from allrank.utils.python_utils import all_equal

C:\Users\mphop\Documents\Jupyter Notebooks\Python exercises\main lightgbm model scripts\final code\final versions\allRank\allrank\utils\file_utils.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import Requirement, resource_filename


In [8]:
import os
from allrank.utils.file_utils import open_local_or_gs
from torchvision.transforms import Compose
from torchvision import transforms
from allrank.data.dataset_loading import FixLength, ToTensor
from allrank.data.dataset_loading import LibSVMDataset


def fix_length_to_longest_slate(ds: LibSVMDataset) -> Compose:
    """
    Helper function returning a transforms.Compose object performing length fixing and tensor conversion.

    Length fixing operation will fix every slate's length to maximum length present in the LibSVMDataset.
    :param ds: LibSVMDataset to transform
    :return: transforms.Compose object
    """
    logger.info("Will pad to the longest slate: {}".format(ds.longest_query_length))
    return transforms.Compose([FixLength(int(ds.longest_query_length)), ToTensor()])

def load_libsvm_role(input_path: str, role: str) -> LibSVMDataset:
    """
    Helper function loading a LibSVMDataset of a specific role.

    The file can be located either in the local filesystem or in GCS.
    :param input_path: LibSVM file directory
    :param role: Dataset role (file name without an extension)
    :return: LibSVMDataset loaded from {input_path}/{role}.txt
    """
    # Normalize the input path for Windows
    input_path = os.path.normpath(input_path)
    input_path = input_path.lstrip("/\\")

    # Build the complete file path
    path = os.path.join(input_path, f"{role}.txt")

    # Normalize again in case input_path or role introduces issues
    path = os.path.normpath(path)

    logger.info(f"Will load {role} data from {path}")

    with open_local_or_gs(path, "rb") as input_stream:
        ds = LibSVMDataset.from_svm_file(input_stream)

    logger.info(f"{role} dataset shape: {ds.shape}")
    return ds


def load_libsvm_dataset_rol(role: str, input_path: str, slate_length: int) -> LibSVMDataset:
    """
    Helper function loading a single role LibSVMDataset.
    
    :param role: The role of the dataset - specifies file name and padding behaviour.
    :param input_path: Directory containing the LibSVM files.
    :param slate_length: Target slate length of the training dataset.
    :return: Loaded LibSVMDataset.
    """

    # Fix the path properly (Windows + Jupyter)
    input_path = os.path.normpath(input_path)

    # Load the dataset
    ds = load_libsvm_role(input_path, role)

    # Apply the correct transform
    if role == "train":
        ds.transform = transforms.Compose([
            FixLength(slate_length),
            ToTensor()
        ])
    else:
        ds.transform = fix_length_to_longest_slate(ds)

    return ds

def parser():
    parser = ArgumentParser("allRank rank and apply click model")
    parser.add_argument("--job-dir", help="Base output path for all experiments", required=True)
    parser.add_argument("--run-id", help="Name of this run to be recorded (must be unique within output dir)",
                        required=True)
    parser.add_argument("--config-file-name", required=True, type=str, help="Name of json file with model config")
    parser.add_argument("--input-model-path", required=True, type=str, help="Path to the model to read weights")
    parser.add_argument("--roles", required=True, type=split_as_strings,
                        help="List of comma-separated dataset roles to load and process")

    return parser


# def run():
    # reproducibility
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)

args = parser().parse_args("--config-file-name scripts/aathletes_listmle_config_listmle_women_combined_colab_config.json --run-id test_run_score --job-dir listmle-women-combined-test-data --input-model-path listmle-women-combined-test-data/results/test_run/model.pkl --roles vali".split(" "))

paths = PathsContainer.from_args(args.job_dir, args.run_id, args.config_file_name)

os.makedirs(paths.base_output_path, exist_ok=True)

create_output_dirs(paths.output_dir)
logger = init_logger(paths.output_dir)

logger.info("will save data in {output_dir}".format(output_dir=paths.base_output_path))

# read config
config = Config.from_json(paths.config_path)
logger.info("Config:\n {}".format(pformat(vars(config), width=1)))

output_config_path = os.path.join(paths.output_dir, "used_config.json")
# execute_command("cp {} {}".format(paths.config_path, output_config_path))

datasets = {role: load_libsvm_dataset_rol(role, config.data.path, config.data.slate_length) for role in args.roles}

n_features = [ds.shape[-1] for ds in datasets.values()]
assert all_equal(n_features), f"Last dimensions of datasets must match but got {n_features}"

# gpu support
dev = get_torch_device()
logger.info("Will use device {}".format(dev.type))

# instantiate model
model = make_model(n_features=n_features[0], **asdict(config.model, recurse=False))

model.load_state_dict(load_state_dict_from_file(args.input_model_path, dev))
logger.info(f"loaded model weights from {args.input_model_path}")

if torch.cuda.device_count() > 1:
    model = CustomDataParallel(model)
    logger.info("Model training will be distributed to {} GPUs.".format(torch.cuda.device_count()))
_ = model.to(dev)


will read config from scripts/aathletes_listmle_config_listmle_women_combined_colab_config.json
[INFO] 2026-08-13 03:58:15 - will save data in listmle-women-combined-test-data
[INFO] 2026-08-13 03:58:15 - Config:
 {'click_model': None,
 'data': DataConfig(path='athletes_women_data_combined', num_workers=10, batch_size=32, slate_length=10, validation_ds_role='vali'),
 'detect_anomaly': True,
 'expected_metrics': {'val': {'ndcg_5': 0.5}},
 'loss': NameArgsConfig(name='listMLE', args={}),
 'lr_scheduler': NameArgsConfig(name='StepLR', args={'step_size': 10, 'gamma': 0.5}),
 'metrics': defaultdict(<class 'list'>,
                        {'ndcg': [5,
                                  8]}),
 'model': ModelConfig(fc_model={'sizes': [64, 128, 256, 128, 64], 'input_norm': True, 'activation': 'ReLU', 'dropout': 0.2}, transformer=None, post_model={'output_activation': None, 'd_output': 1}),
 'optimizer': NameArgsConfig(name='Adam', args={'lr': 0.005}),
 'training': TrainingConfig(epochs=40, gradi

In [9]:
from typing import Tuple, Dict, List, Generator

import torch
from torch.utils.data.dataloader import DataLoader

import allrank.models.losses as losses
from allrank.config import Config
from allrank.data.dataset_loading import LibSVMDataset
from allrank.models.metrics import ndcg, dcg
from allrank.models.model import LTRModel
from allrank.models.model_utils import get_torch_device


def score_datasets(datasets: Dict[str, LibSVMDataset], model: LTRModel, config: Config) \
        -> Dict[str, Tuple[torch.Tensor]]:
    """
    Ranks given datasets according to a given model

    :param datasets: dictionary of role -> dataset that will be ranked
    :param model: a model to use for scoring documents
    :param config: config for DataLoaders
    :return: dictionary of role -> ranked dataset
        every dataset is a Tuple of torch.Tensor - storing X and y in the descending order of the scores.
    """

    dataloaders = {role: __create_data_loader(ds, config) for role, ds in datasets.items()}

    scored_slates = {role: __score_slates(dl, model) for role, dl in dataloaders.items()}

    return scored_slates


def __create_data_loader(ds: LibSVMDataset, config: Config) -> DataLoader:
    return DataLoader(ds, batch_size=config.data.batch_size, num_workers=config.data.num_workers, shuffle=False)


def __score_slates(dataloader: DataLoader, model: LTRModel) -> Tuple[torch.Tensor, torch.Tensor]:
    all_scores = []
    
    model.eval()
    device = get_torch_device()
    with torch.no_grad():
        for xb, yb, _ in dataloader:
            X = xb.type(torch.float32).to(device=device)
            y_true = yb.to(device=device)

            input_indices = torch.ones_like(y_true).type(torch.long)
            mask = (y_true == losses.PADDED_Y_VALUE)
            scores = model.score(X, mask, input_indices)

            scores[mask] = float('-inf')

            all_scores.append(scores.cpu())
                        
    return torch.cat(all_scores)


scored_slates = score_datasets(datasets, model, config)

C:\Users\mphop\anaconda3\envs\allrank-env\lib\site-packages\torch\utils\data\dataloader.py:627: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 8 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [10]:
scored_slates['vali'].shape

torch.Size([8050, 10])

In [11]:
scored_slates['vali']

tensor([[0.8198, 0.8198,   -inf,  ...,   -inf,   -inf,   -inf],
        [0.8198, 0.8198,   -inf,  ...,   -inf,   -inf,   -inf],
        [0.8198, 0.8198,   -inf,  ...,   -inf,   -inf,   -inf],
        ...,
        [0.8198, 0.8198, 0.8198,  ...,   -inf,   -inf,   -inf],
        [0.8198, 0.8198,   -inf,  ...,   -inf,   -inf,   -inf],
        [0.8198, 0.8198,   -inf,  ...,   -inf,   -inf,   -inf]])

In [12]:
# Save tensor to a text file line by line
with open("allrank_listmle_women_combined_tensor_output.txt", "w") as f:
    for row in scored_slates['vali']:
        row_str = " ".join(map(str, row.tolist()))  # Convert tensor row to space-separated string
        f.write(row_str + "\n")  # Write each row on a new line